In [0]:
gold_path='/Volumes/main/retail/data/gold/online_retail'

In [0]:
# List of paths
paths = {
    "Gold_Sales":f"{gold_path}/Complete_Data",
    "Country_Sales":f"{gold_path}/Country_Sales",
    "Product_Performance":f"{gold_path}/Product_Performance",
    "Sales_Summary":f"{gold_path}/Sales_Summary",
    "Customer_Sales":f"{gold_path}/Customer_Sales"
}

# Loop through each path and verify record count
for view_name,path in paths.items():
    try:
        # Read data (change format if needed: parquet/csv/json/delta)
        df = spark.read.parquet(path)
        df.createOrReplaceTempView(view_name)
        print(f"View {view_name} created with {df.count()} records.")

    except Exception as e:
        print(f"Error reading path: {path}")
        print(f"Error: {str(e)}")
        print("-" * 50)


View Gold_Sales created with 392732 records.
View Country_Sales created with 37 records.
View Product_Performance created with 3897 records.
View Sales_Summary created with 1 records.
View Customer_Sales created with 4339 records.


In [0]:
%sql
SELECT
    year(Invoice_Dt) AS year,
    month(Invoice_Dt) AS month,
    ROUND(SUM(Total_Amt), 2) AS revenue
FROM Gold_Sales
GROUP BY year, month
ORDER BY year, month;



year,month,revenue
2010,12,570422.73
2011,1,568101.31
2011,2,446084.92
2011,3,594081.76
2011,4,468374.33
2011,5,677355.15
2011,6,660046.05
2011,7,598962.9
2011,8,644051.04
2011,9,950690.2


In [0]:
%sql

SELECT Country,description as product_desc,revenue from (SELECT
    country,
    description,
    Total_Amt as revenue,
    dense_rank() OVER (
        PARTITION BY country
        ORDER BY Total_Amt DESC
    ) AS product_rank
FROM Gold_Sales)t1
where product_rank=1
order by revenue desc;

Country,product_desc,revenue
United Kingdom,"PAPER CRAFT , LITTLE BIRDIE",168469.6
Netherlands,RABBIT NIGHT LIGHT,4992.0
France,Manual,4161.06
France,Manual,4161.06
Singapore,Manual,3949.32
Japan,ROUND SNACK BOXES SET OF 4 FRUITS,3794.4
EIRE,RED RETROSPOT CAKE STAND,2365.2
Australia,RABBIT NIGHT LIGHT,1718.4
Spain,CHILDRENS CUTLERY POLKADOT PINK,1350.0
Spain,CHILDRENS CUTLERY POLKADOT BLUE,1350.0


In [0]:
%sql
SELECT * from (SELECT
    country,
    cust_id,
    total_amt as total_spend,
    round(AVG(total_amt) OVER (
        PARTITION BY country
    ),2) AS avg_country_spend,
    dense_rank() over(partition by country order by total_amt desc) as rnk
FROM Gold_Sales)t1
where rnk<=3;

country,cust_id,total_spend,avg_country_spend,rnk
Australia,12415.0,1718.4,116.94,1
Australia,12415.0,1260.0,116.94,2
Australia,12415.0,1074.0,116.94,3
Austria,12818.0,360.0,25.62,1
Austria,12818.0,302.4,25.62,2
Austria,12818.0,302.4,25.62,2
Austria,12360.0,160.0,25.62,3
Austria,12358.0,160.0,25.62,3
Bahrain,12355.0,120.0,32.26,1
Bahrain,12355.0,75.0,32.26,2


In [0]:
%sql

WITH Monthly_Sales AS (
    SELECT
        YEAR(Invoice_Dt) AS year,
        MONTH(Invoice_Dt) AS month,
        ROUND(SUM(Total_Amt), 2) AS revenue
    FROM Gold_Sales
    GROUP BY YEAR(Invoice_Dt), MONTH(Invoice_Dt)
)
SELECT
    year,
    month,
    revenue,
    LAG(revenue) OVER (
        ORDER BY year, month
    ) AS previous_month_revenue,
    revenue - LAG(revenue) OVER (
        ORDER BY year, month
    ) AS revenue_difference
FROM Monthly_Sales;

year,month,revenue,previous_month_revenue,revenue_difference
2010,12,570422.73,null,null
2011,1,568101.31,570422.73,-2321.4199999999255
2011,2,446084.92,568101.31,-122016.39000000007
2011,3,594081.76,446084.92,147996.84000000003
2011,4,468374.33,594081.76,-125707.43
2011,5,677355.15,468374.33,208980.82
2011,6,660046.05,677355.15,-17309.099999999977
2011,7,598962.9,660046.05,-61083.15000000002
2011,8,644051.04,598962.9,45088.140000000014
2011,9,950690.2,644051.04,306639.1599999999
